# Ecommerce Analytics — Data Exploration

This notebook explores the raw (bronze) data generated by `hakio_data_project.py` and documents the data quality issues that informed the transform and validation rules in the SQL pipeline.

**Pipeline overview:** `bronze → transform (validate) → data_quality → silver → gold`

## 1. Import Required Libraries

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

con = duckdb.connect("data/analytics.duckdb", read_only=True)
print("Connected to analytics.duckdb")

## 2. Load the Dataset

List all schemas and tables in the database, then load each bronze table into a DataFrame.

In [ ]:
# Show all schemas and tables
con.sql("""
    SELECT table_schema, table_name, table_type
    FROM information_schema.tables
    ORDER BY table_schema, table_name
""").show()

In [ ]:
# Load bronze tables into DataFrames
df_products = con.sql("SELECT * FROM bronze.products").df()
df_orders = con.sql("SELECT * FROM bronze.orders").df()
df_order_lines = con.sql("SELECT * FROM bronze.order_lines").df()
df_refunds = con.sql("SELECT * FROM bronze.refunds").df()

print(f"bronze.products:    {len(df_products):,} rows, {len(df_products.columns)} columns")
print(f"bronze.orders:      {len(df_orders):,} rows, {len(df_orders.columns)} columns")
print(f"bronze.order_lines: {len(df_order_lines):,} rows, {len(df_order_lines.columns)} columns")
print(f"bronze.refunds:     {len(df_refunds):,} rows, {len(df_refunds.columns)} columns")

## 3. Initial Data Inspection

First look at each table's structure and sample rows.

In [ ]:
df_products.head(10)

In [ ]:
df_orders.head(10)

In [ ]:
df_order_lines.head(10)

In [ ]:
df_refunds.head(10)

## 4. Data Types and Schema Overview

All bronze columns are loaded as `VARCHAR` (`all_varchar=true`) to preserve raw data. This section shows column info for each table.

In [ ]:
for name, df in [("products", df_products), ("orders", df_orders),
                  ("order_lines", df_order_lines), ("refunds", df_refunds)]:
    print(f"\n{'='*60}")
    print(f"bronze.{name} — {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"{'='*60}")
    print(df.dtypes.to_string())

## 5. Missing Values Analysis

Check for NULLs across all columns. Due to schema drift (`union_by_name=true`), some columns only appear in certain daily files — leading to NULLs where the column wasn't present in the source CSV.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Missing Values (%) per Column — Bronze Tables", fontsize=14)

for ax, (name, df) in zip(axes.flat, [("products", df_products), ("orders", df_orders),
                                        ("order_lines", df_order_lines), ("refunds", df_refunds)]):
    null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]
    if len(null_pct) > 0:
        null_pct.plot.barh(ax=ax, color="salmon")
        ax.set_xlabel("% Missing")
    else:
        ax.text(0.5, 0.5, "No missing values", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(f"bronze.{name}")

plt.tight_layout()
plt.show()

## 6. Descriptive Statistics

Summary statistics for numeric-like columns. Since bronze stores everything as `VARCHAR`, we'll cast and describe the key numeric fields.

In [ ]:
# Products: cost_price and list_price distributions
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(TRY_CAST(cost_price AS DECIMAL(10,2))) AS valid_cost_price,
        COUNT(*) - COUNT(TRY_CAST(cost_price AS DECIMAL(10,2))) AS invalid_cost_price,
        MIN(TRY_CAST(cost_price AS DECIMAL(10,2))) AS min_cost,
        AVG(TRY_CAST(cost_price AS DECIMAL(10,2))) AS avg_cost,
        MAX(TRY_CAST(cost_price AS DECIMAL(10,2))) AS max_cost,
        COUNT(TRY_CAST(list_price AS DECIMAL(10,2))) AS valid_list_price,
        COUNT(*) - COUNT(TRY_CAST(list_price AS DECIMAL(10,2))) AS invalid_list_price,
        MIN(TRY_CAST(list_price AS DECIMAL(10,2))) AS min_list,
        MAX(TRY_CAST(list_price AS DECIMAL(10,2))) AS max_list
    FROM bronze.products
""").show()

In [ ]:
# Orders: categorical distributions
print("Order status distribution:")
print(df_orders["order_status"].value_counts().to_string())
print(f"\nPayment methods:")
print(df_orders["payment_method"].value_counts().to_string())
print(f"\nCountries:")
print(df_orders["country"].value_counts().to_string())

## 7. Duplicate Detection

The data generator intentionally injects ~4-5% duplicate rows per day per table. Let's measure the actual duplication rate.

In [ ]:
con.sql("""
    SELECT 'products' AS table_name,
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_rows,
        ROUND((COUNT(*) - COUNT(DISTINCT product_id))::FLOAT / COUNT(*) * 100, 1) AS dup_pct
    FROM bronze.products
    UNION ALL
    SELECT 'orders',
        COUNT(*),
        COUNT(*) - COUNT(DISTINCT order_id),
        ROUND((COUNT(*) - COUNT(DISTINCT order_id))::FLOAT / COUNT(*) * 100, 1)
    FROM bronze.orders
    UNION ALL
    SELECT 'order_lines',
        COUNT(*),
        COUNT(*) - COUNT(DISTINCT order_line_id),
        ROUND((COUNT(*) - COUNT(DISTINCT order_line_id))::FLOAT / COUNT(*) * 100, 1)
    FROM bronze.order_lines
    UNION ALL
    SELECT 'refunds',
        COUNT(*),
        COUNT(*) - COUNT(DISTINCT refund_id),
        ROUND((COUNT(*) - COUNT(DISTINCT refund_id))::FLOAT / COUNT(*) * 100, 1)
    FROM bronze.refunds
""").show()

## 8. Outlier Detection and Visualization

Look at numeric field distributions to spot injected bad records (negative prices, extreme values).

In [ ]:
# Cast numeric fields for visualization (invalid values become NaN)
products_numeric = con.sql("""
    SELECT TRY_CAST(cost_price AS FLOAT) AS cost_price,
           TRY_CAST(list_price AS FLOAT) AS list_price
    FROM bronze.products
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
products_numeric.boxplot(column="cost_price", ax=axes[0])
axes[0].set_title("Products: cost_price")
products_numeric.boxplot(column="list_price", ax=axes[1])
axes[1].set_title("Products: list_price (note negative outlier)")
plt.tight_layout()
plt.show()

In [ ]:
# Order lines: unit_price and quantity outliers
order_lines_numeric = con.sql("""
    SELECT TRY_CAST(quantity AS INT) AS quantity,
           TRY_CAST(unit_price AS FLOAT) AS unit_price,
           TRY_CAST(line_discount_amount AS FLOAT) AS line_discount_amount
    FROM bronze.order_lines
""").df()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
order_lines_numeric.boxplot(column="quantity", ax=axes[0])
axes[0].set_title("Order Lines: quantity")
order_lines_numeric.boxplot(column="unit_price", ax=axes[1])
axes[1].set_title("Order Lines: unit_price")
order_lines_numeric.boxplot(column="line_discount_amount", ax=axes[2])
axes[2].set_title("Order Lines: line_discount_amount")
plt.tight_layout()
plt.show()

## 9. Value Distribution Analysis

Distribution of key categorical and numeric fields across the bronze data.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Product types
df_products["product_type"].value_counts().plot.bar(ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Product Type Distribution")
axes[0, 0].tick_params(axis="x", rotation=45)

# Order status
df_orders["order_status"].value_counts().plot.bar(ax=axes[0, 1], color="coral")
axes[0, 1].set_title("Order Status Distribution")
axes[0, 1].tick_params(axis="x", rotation=45)

# Payment methods
df_orders["payment_method"].value_counts().plot.bar(ax=axes[1, 0], color="mediumseagreen")
axes[1, 0].set_title("Payment Method Distribution")
axes[1, 0].tick_params(axis="x", rotation=45)

# Country
df_orders["country"].value_counts().plot.bar(ax=axes[1, 1], color="mediumpurple")
axes[1, 1].set_title("Orders by Country")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 10. Data Transformation — Bad Records Found

These are the specific data quality issues that were discovered during exploration and led to the validation rules in the `transform` layer.

**Issues found:**
- **Products:** `product_id` not starting with 'P' (empty IDs), `list_price` not castable to float (`not_a_number`, missing values)
- **Orders:** `order_id` 2nd character not numeric (`O-BAD-001`), `order_ts` not a valid timestamp (`not-a-timestamp`)
- **Order Lines:** `order_line_id` 2nd character not numeric (`L-BAD-ORPHAN`), `unit_price` not castable to float (`NaN`)
- **Refunds:** `refund_id` 2nd character not numeric (`R-BAD-001`), `refund_ts` not a valid timestamp (`32/13/2026`)

In [ ]:
# Sample bad records from each table
print("=== Bad Products (invalid product_id or list_price) ===")
con.sql("""
    SELECT product_id, sku, product_name, cost_price, list_price
    FROM bronze.products
    WHERE SUBSTRING(product_id, 1, 1) != 'P' OR product_id IS NULL OR product_id = ''
       OR TRY_CAST(list_price AS FLOAT) IS NULL
    LIMIT 10
""").show()

print("\n=== Bad Orders (invalid order_id or timestamp) ===")
con.sql("""
    SELECT order_id, customer_id, order_ts, country, currency
    FROM bronze.orders
    WHERE TRY_CAST(SUBSTRING(order_id, 2, 1) AS INT) IS NULL
       OR TRY_CAST(order_ts AS DATETIME) IS NULL
    LIMIT 10
""").show()

print("\n=== Bad Order Lines (invalid order_line_id or unit_price) ===")
con.sql("""
    SELECT order_line_id, order_id, product_id, quantity, unit_price
    FROM bronze.order_lines
    WHERE TRY_CAST(SUBSTRING(order_line_id, 2, 1) AS INT) IS NULL
       OR TRY_CAST(unit_price AS FLOAT) IS NULL
    LIMIT 10
""").show()

print("\n=== Bad Refunds (invalid refund_id or timestamp) ===")
con.sql("""
    SELECT refund_id, order_line_id, refund_ts, refund_amount
    FROM bronze.refunds
    WHERE TRY_CAST(SUBSTRING(refund_id, 2, 1) AS INT) IS NULL
       OR TRY_CAST(refund_ts AS DATETIME) IS NULL
    LIMIT 10
""").show()

## 11. Data Validation — Quality Summary

Results from the `data_quality` schema, showing how many records pass vs fail validation in each table.

In [ ]:
print("=== Data Quality: Products ===")
con.sql("SELECT * FROM data_quality.products ORDER BY error_count DESC").show()

print("=== Data Quality: Orders ===")
con.sql("SELECT * FROM data_quality.orders ORDER BY error_count DESC").show()

print("=== Data Quality: Order Lines ===")
con.sql("SELECT * FROM data_quality.order_lines ORDER BY error_count DESC").show()

print("=== Data Quality: Refunds ===")
con.sql("SELECT * FROM data_quality.refunds ORDER BY error_count DESC").show()

In [ ]:
# Visualize error distribution across all tables
dq = con.sql("""
    SELECT 'products' AS table_name, error_reason, error_count FROM data_quality.products
    UNION ALL
    SELECT 'orders', error_reason, error_count FROM data_quality.orders
    UNION ALL
    SELECT 'order_lines', error_reason, error_count FROM data_quality.order_lines
    UNION ALL
    SELECT 'refunds', error_reason, error_count FROM data_quality.refunds
""").df()

fig, ax = plt.subplots(figsize=(12, 6))
pivot = dq.pivot_table(index="table_name", columns="error_reason", values="error_count", fill_value=0)
pivot.plot.bar(ax=ax, stacked=True)
ax.set_title("Data Quality: Error Counts by Table and Reason")
ax.set_ylabel("Row Count")
ax.legend(title="Error Reason", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 12. Bronze vs Silver — Pipeline Validation

Compare row counts between bronze (raw) and silver (cleaned, deduped, validated) to confirm the pipeline is filtering correctly.

In [ ]:
comparison = con.sql("""
    SELECT 'products' AS table_name,
        (SELECT COUNT(*) FROM bronze.products) AS bronze_rows,
        (SELECT COUNT(*) FROM silver.products) AS silver_rows,
        (SELECT COUNT(*) FROM bronze.products) - (SELECT COUNT(*) FROM silver.products) AS rows_removed
    UNION ALL
    SELECT 'orders',
        (SELECT COUNT(*) FROM bronze.orders),
        (SELECT COUNT(*) FROM silver.orders),
        (SELECT COUNT(*) FROM bronze.orders) - (SELECT COUNT(*) FROM silver.orders)
    UNION ALL
    SELECT 'order_lines',
        (SELECT COUNT(*) FROM bronze.order_lines),
        (SELECT COUNT(*) FROM silver.order_lines),
        (SELECT COUNT(*) FROM bronze.order_lines) - (SELECT COUNT(*) FROM silver.order_lines)
    UNION ALL
    SELECT 'refunds',
        (SELECT COUNT(*) FROM bronze.refunds),
        (SELECT COUNT(*) FROM silver.refunds),
        (SELECT COUNT(*) FROM bronze.refunds) - (SELECT COUNT(*) FROM silver.refunds)
""").df()

display(comparison)

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison))
width = 0.35
ax.bar(x - width/2, comparison["bronze_rows"], width, label="Bronze (raw)", color="sienna")
ax.bar(x + width/2, comparison["silver_rows"], width, label="Silver (clean)", color="silver")
ax.set_xticks(x)
ax.set_xticklabels(comparison["table_name"])
ax.set_ylabel("Row Count")
ax.set_title("Bronze vs Silver Row Counts")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
con.close()
print("Connection closed.")